# 🛠️ Lab 2: Give It Hands — Build the Agent Loop From Scratch

No frameworks. No magic. In ~30 lines of Python you will build the **Think → Act → Observe** loop that powers every AI agent on Earth — and watch the model that failed yesterday's multiplication **nail it** with a calculator tool.

## 📚 The plan
1. 🧰 Define two tools (they're just Python functions!)
2. 📜 Teach the model the tool protocol (JSON — yesterday's structured output trick)
3. 🔁 Build the loop and watch it think live
4. 🧗 Multi-step missions + guardrails

🔑 Needs your OpenRouter API key.

In [ ]:
!pip install openai requests

### 🛠️ Setup

In [ ]:
!pip install openai sentence-transformers -q

import json
from openai import OpenAI
from getpass import getpass

api_key = getpass("🔑 Paste your OpenRouter API key: ")
client = OpenAI(api_key=api_key)

In [ ]:
MODEL = "gpt-5.4-nano"

def chat(messages, temperature=0):
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
    return r.choices[0].message.content

print("🛠️ Workshop open. Let's build hands.")

### 🧰 Section 1: Tools Are Just Python Functions

That's the demystifying truth: a "tool" is any function we choose to expose. Today: a **calculator** and a (mock) **weather service**. *(In real systems the weather tool would call a live API — same shape, real data.)*

In [ ]:
import requests

def calculator(expression):
    """Evaluate a math expression like '847293 * 652'. Digits and operators only."""
    allowed = set('0123456789+-*/(). ')
    if not set(expression) <= allowed:
        return "ERROR: only numbers and + - * / ( ) are allowed."
    try:
        return str(eval(expression))   # safe here because we whitelisted the characters
    except Exception as e:
        return f"ERROR: {e}"

def get_weather(city):
    """Real weather via the free Open-Meteo API (no API key needed)."""
    try:
        # Step 1: turn the city name into coordinates (geocoding)
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city.strip(), "count": 1},
            timeout=10,
        ).json()
        if not geo.get("results"):
            return f"ERROR: couldn't find a city called '{city}'."
        loc = geo["results"][0]
        lat, lon = loc["latitude"], loc["longitude"]

        # Step 2: get current weather at those coordinates
        w = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={"latitude": lat, "longitude": lon, "current": "temperature_2m,wind_speed_10m"},
            timeout=10,
        ).json()
        cur = w["current"]
        return (f"{loc['name']}, {loc.get('country', '')}: "
                f"{cur['temperature_2m']}°C, wind {cur['wind_speed_10m']} km/h")
    except Exception as e:
        return f"ERROR: weather lookup failed ({e})."

In [ ]:
TOOLS = {"calculator": calculator, "get_weather": get_weather}

print(calculator("847293 * 652"))     # the answer the LLM couldn't compute yesterday!
print(get_weather("Khobar"))           # now REAL, live data
print(get_weather("Tokyo"))            # works for any city on Earth

### 📜 Section 2: The Protocol — Teaching the Model to Order From the Kitchen

The LLM can only output **text**. So we make a deal with it (in the system prompt): reply ONLY in JSON, in one of two shapes — a **tool request**, or a **final answer**. This is exactly the structured-output trick from Day 4 Lab 3.

In [ ]:
import json
def parse_json(text):
    text = text.replace('```json', '').replace('```', '').strip()
    start = text.find('{')
    obj, _ = json.JSONDecoder().raw_decode(text[start:])
    return obj

In [ ]:
AGENT_SYSTEM = """You are an agent that solves tasks step by step using tools.

Available tools:
- calculator: evaluates a math expression. Input example: "12 * (3 + 4)"
- get_weather: returns current weather for a city. Input example: "Dammam"

You must reply with ONLY a JSON object, no other text, in ONE of these two shapes:
1. To use a tool:      {"thought": "<why you need it>", "tool": "<tool name>", "input": "<tool input>"}
2. To finish:          {"thought": "<your reasoning>", "answer": "<final answer for the user>"}

Use one tool at a time. After each tool, you will receive its result and can decide your next step.
Never invent tool results — always call the tool."""

# Let's peek at ONE raw step before building the loop:
messages = [{"role": "system", "content": AGENT_SYSTEM},
            {"role": "user", "content": "What is 847293 * 652?"}]
print(parse_json(chat(messages)))
# 👀 It didn't answer — it ORDERED. That JSON is the model asking OUR code to run the calculator.

### 🔁 Section 3: The Loop — 30 Lines That Power the AI Industry

Now we close the circuit: parse the JSON → run the tool → feed the result back → repeat until the model outputs an `answer`. Watch every step print live.

In [ ]:
def parse_json(text):
    """Extract the FIRST JSON object from the model's reply (ignores duplicates)."""
    text = text.replace('```json', '').replace('```', '').strip()
    start = text.find('{')
    obj, _ = json.JSONDecoder().raw_decode(text[start:])
    return obj

def run_agent(task, max_steps=5, verbose=True):
    messages = [{"role": "system", "content": AGENT_SYSTEM},
                {"role": "user", "content": task}]

    for step in range(1, max_steps + 1):
        reply = chat(messages)
        try:
            action = parse_json(reply)
        except Exception as e:
            print(f"⚠️ parse failed: {e} | raw reply: {reply[:120]}")
            messages.append({"role": "user", "content": "Your reply was not valid JSON. Reply with ONLY the JSON object."})
            continue

        if verbose:
            print(f"\n——— Step {step} ———")
            print(f"💭 THINK: {action.get('thought', '')}")

        # Case 1: the model is done
        if 'answer' in action:
            if verbose: print(f"✅ ANSWER: {action['answer']}")
            return action['answer']

        # Case 2: the model wants a tool
        tool_name, tool_input = action.get('tool'), action.get('input', '')
        if tool_name not in TOOLS:
            result = f"ERROR: unknown tool '{tool_name}'. Available: {list(TOOLS)}"
        else:
            result = TOOLS[tool_name](tool_input)

        if verbose:
            print(f"🔧 ACT:     {tool_name}({tool_input})")
            print(f"👁️ OBSERVE: {result}")

        # Feed the observation back — THIS is what makes it a loop
        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": f"Tool result: {result}"})

    return "⚠️ Stopped: reached max_steps without a final answer."

In [ ]:
run_agent("What is 847293 * 652?")

**Read that trace.** Yesterday this exact model *guessed* wrong digits. Today it thought, called the calculator, observed the true result, and answered correctly. **Same brain — new hands.** 🤯

### 🧗 Section 4: Multi-Step Missions — Watch It Plan

In [ ]:
# This needs TWO different tools, in the right order. Nobody tells it the order — it decides.
run_agent("Take the temperature in Riyadh (just the number in Celsius), multiply it by 3, and subtract 7. Show the final number.")

In [ ]:
# Three tool calls minimum. Count the steps!
run_agent("Compare the current temperatures of Dammam and Abha, and tell me the difference in degrees.")

### 🎯 Mini Exercise 4.1 — Error Recovery 😈
Ask the agent about the weather in a city NOT in our mock data (e.g., Tokyo). Watch the trace: it should observe the ERROR message and react. Does it apologize? Try another spelling? Give up gracefully? **An agent that handles failed tools is what separates a demo from a product.**

In [ ]:
# TO-DO: run_agent with a city we don't have, and study the trace


### 🎯 Mini Exercise 4.2 — Build Your OWN Tool 🔨
Add a third tool and register it in `TOOLS` **and** describe it in `AGENT_SYSTEM` (the agent only knows what the system prompt tells it!). Ideas:
- `count_letters(text)` — the famous "how many r's in strawberry" fixer 🍓
- `convert_currency(amount_sar)` — SAR → USD with a fixed rate
- `roll_dice(sides)` — for the gamers 🎲

Then give the agent a task that needs it!

In [ ]:
# TO-DO: 1) define your function  2) add to TOOLS  3) describe it inside AGENT_SYSTEM  4) test!

# --- 1. Define the new tool functions ---

def convert_currency(amount_usd):
    """Convert US Dollars to Saudi Riyals at a fixed rate. Input: a number of US Dollars."""
    try:
        usd = float(amount_usd)
        sar = 
        return f"{usd} USD = {sar} SAR (rate: 1 USD = _____ SAR)."
    except Exception:
        return "ERROR: input must be a number, e.g. '2000'."


In [ ]:
# --- 2. Register them in TOOLS (so the loop can actually call them) ---
TOOLS["convert_currency"] = 
print("🧰 Toolbox now:", list(TOOLS))

In [ ]:
# --- 3. Describe them in AGENT_SYSTEM (so the model KNOWS they exist!) ---

AGENT_SYSTEM = """You are an agent that solves tasks step by step using tools.

Available tools:
- calculator: evaluates a math expression. Input example: "12 * (3 + 4)"
- get_weather: returns current weather for a city. Input example: "Dammam"
- 



You must reply with ONLY a JSON object, no other text, in ONE of these two shapes:
1. To use a tool:      {"thought": "<why you need it>", "tool": "<tool name>", "input": "<tool input>"}
2. To finish:          {"thought": "<your reasoning>", "answer": "<final answer for the user>"}

Use one tool at a time. After each tool, you will receive its result and can decide your next step.
Output exactly ONE JSON object. Never invent tool results — always call the tool."""


In [ ]:
# --- 4. Test each new tool ---

print("\n" + "="*60)
run_agent("I have 3500 dollars. How many riyals is that?")

## The GOLDEN Question 🏆

Look at your `run_agent` code and find every **guardrail** hiding in it. There are at least three: something that stops infinite loops, something that handles a hallucinated tool name, and something that handles broken JSON.

**Name all three, point to the exact line, and explain what disaster each one prevents. Then: your agent is 30 lines — what would you add before trusting it with a `send_email` tool?** 🤔